In [1]:
# Imports
import sys
from pathlib import Path
repo_root = Path.cwd().parents[0]
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import copy
import pprint
from collections.abc import Mapping

import equinox as eqx
import jax
import jax.numpy as jnp
import jax.tree_util as jtu
import optax

from darnax.datasets.classification.mnist import Mnist
from darnax.datasets.classification.cifar10 import Cifar10
from darnax.layer_maps.sparse import LayerMap
from darnax.modules.fully_connected import (
    FullyConnected,
    FrozenRescaledFullyConnected,
    SparseFullyConnected,
)
from darnax.modules.input_output import OutputLayer
from darnax.modules.recurrent import SparseRecurrentDiscrete
from darnax.orchestrators.sequential import SequentialOrchestrator
from darnax.states.sequential import SequentialState
from darnax.trainers.dynamical import DynamicalTrainer
from darnax.utils.typing import PyTree

# conv adapters (my file)
from darnax.modules.conv.conv_adapters import ConvAdapter, ConvRecurrentDiscrete


# Conv experiment notebook (experiments/conv_experiment.ipynb)


In [18]:
# Params (short run)
PARAMS = {
    "master_seed": 0,
    "epochs": 2,
    "model": {
        "kwargs": {
            "seed": 44,
            "dim_data": 3072,
            # For conv: set dim_hidden as (H, W, C)
            "dim_hidden": (32, 32, 8),
            "num_labels": 10,
            "sparsity": 0.99,
            "sparsity_win": 0.9,
            "strength_forth": 5.0,
            "strength_back": 1.3,
            "j_d": 0.95,
            "threshold_in": 1.78,
            "threshold_out": 7.0,
            "threshold_j": 1.78,
        }
    },
    "data": {"kwargs": {"batch_size": 16, "linear_projection": None}},  # CIFAR10: no 'flatten', use linear_projection=None
    "optimizer": {
        "learning_rate_win": 0.159,
        "learning_rate_j": 0.058,
        "learning_rate_wout": 0.17,
        "weight_decay_win": 0.01,
        "weight_decay_j": 0.00006,
        "weight_decay_wout": 0.02,
    },
    "trainer": {"kwargs": {"warmup_n_iter": 1, "train_clamped_n_iter": 3, "train_free_n_iter": 3, "eval_n_iter": 3}},
}

pprint.pprint(PARAMS)

{'data': {'kwargs': {'batch_size': 16, 'linear_projection': None}},
 'epochs': 2,
 'master_seed': 0,
 'model': {'kwargs': {'dim_data': 3072,
                      'dim_hidden': (32, 32, 8),
                      'j_d': 0.95,
                      'num_labels': 10,
                      'seed': 44,
                      'sparsity': 0.99,
                      'sparsity_win': 0.9,
                      'strength_back': 1.3,
                      'strength_forth': 5.0,
                      'threshold_in': 1.78,
                      'threshold_j': 1.78,
                      'threshold_out': 7.0}},
 'optimizer': {'learning_rate_j': 0.058,
               'learning_rate_win': 0.159,
               'learning_rate_wout': 0.17,
               'weight_decay_j': 6e-05,
               'weight_decay_win': 0.01,
               'weight_decay_wout': 0.02},
 'trainer': {'kwargs': {'eval_n_iter': 3,
                        'train_clamped_n_iter': 3,
                        'train_free_n_iter': 3,
    

In [23]:
# Local build_model that supports conv adapters (flat-interface)
from darnax.layer_maps.sparse import LayerMap
from darnax.modules.fully_connected import SparseFullyConnected, FrozenRescaledFullyConnected, FullyConnected

import math

def build_model(
    seed: int,
    dim_data: int,
    dim_hidden,
    sparsity: float,
    sparsity_win: float,
    num_labels: int,
    strength_forth: float,
    strength_back: float,
    threshold_in: float,
    threshold_out: float,
    threshold_j: float,
    j_d: float,
    win_type: str = 'conv',
    j_type: str = 'conv',
    kernel_size: int = 3,
    c_in: int = 1,  # Input channels (1 for MNIST, 3 for CIFAR10)
):
    # dim_hidden can be int or tuple (H,W,C)
    if isinstance(dim_hidden, (tuple, list)):
        h_hidden, w_hidden, c_hidden = dim_hidden
        dim_hidden_flat = h_hidden * w_hidden * c_hidden
    else:
        dim_hidden_flat = dim_hidden
        # infer spatial dims as sqrt if possible
        side = int(math.sqrt(dim_hidden_flat))
        h_hidden, w_hidden, c_hidden = side, side, 1

    state = SequentialState((dim_data, dim_hidden_flat, num_labels))

    master_key = jax.random.PRNGKey(seed)
    keys = jax.random.split(master_key, num=5)

    # Win
    if win_type == 'fc':
        win = SparseFullyConnected(in_features=dim_data, out_features=dim_hidden_flat, strength=strength_forth, threshold=threshold_in, sparsity=sparsity_win, key=keys[0])
    else:
        # Compute spatial dimensions from dim_data and c_in
        spatial_dim = int((dim_data / c_in) ** 0.5)
        win = ConvAdapter(h_in=spatial_dim, w_in=spatial_dim, c_in=c_in, c_out=c_hidden, kernel_size=kernel_size, h_out=h_hidden, w_out=w_hidden, strength=strength_forth, threshold=threshold_in, key=keys[0])

    # J
    if j_type == 'fc':
        j_mod = SparseRecurrentDiscrete(features=dim_hidden_flat, j_d=j_d, sparsity=sparsity, threshold=threshold_j, key=keys[1])
    else:
        j_mod = ConvRecurrentDiscrete(h=h_hidden, w=w_hidden, channels=c_hidden, kernel_size=kernel_size, j_d=j_d, threshold=threshold_j, key=keys[1])

    feedback = FrozenRescaledFullyConnected(in_features=num_labels, out_features=dim_hidden_flat, strength=strength_back, threshold=0.0, key=keys[2])
    output = FullyConnected(in_features=dim_hidden_flat, out_features=num_labels, strength=1.0, threshold=threshold_out, key=keys[3])

    layer_map = {
        1: {0: win, 1: j_mod, 2: feedback},
        2: {1: output, 2: OutputLayer()},
    }
    lmap = LayerMap.from_dict(layer_map)
    orch = SequentialOrchestrator(lmap)
    return state, orch

In [20]:
def make_lr_map_v2(
    model: SequentialOrchestrator,
    overrides: Mapping[tuple[int, int], str] | None = None,
    default_label: str = "default",
) -> PyTree:
    """Build a PyTree of parameter labels for ``optax.multi_transform``.

    Parameters
    ----------
    model:
        Orchestrator model (must have a ``.lmap`` field).
    overrides:
        Optional mapping from ``(layer_idx, position_idx)`` to a label string.
        Example: ``{(1, 0): "w_in", (1, 1): "j", (2, 1): "w_out"}``.
    default_label:
        Label assigned to all parameters not specified in ``overrides``.

    Returns
    -------
    labels:
        PyTree matching the parameter structure, each leaf a string label.

    """
    params, _ = eqx.partition(model, eqx.is_inexact_array)

    def like(tree, value: str):
        """Broadcast a scalar label to a tree with the same structure."""
        return jtu.tree_map(lambda _: value, tree, is_leaf=eqx.is_array)

    labels = jtu.tree_map(lambda _: default_label, params, is_leaf=eqx.is_array)

    if overrides:
        for (i, j), label in overrides.items():
            labels = eqx.tree_at(
                lambda m: m.lmap[i][j],
                labels,
                replace=like(params.lmap[i][j], label),
            )

    return labels

In [21]:
# Local decay handling conv kernels and FC matrices
import equinox as eqx

def decay(orchestrator: SequentialOrchestrator, cfg: dict):
    new_orch = orchestrator
    # flatted dim_hidden handling
    dim_hidden = cfg["model"]["kwargs"]["dim_hidden"]
    if isinstance(dim_hidden, (tuple, list)):
        h, w, c = dim_hidden
        dim_hidden_flat = h * w * c
    else:
        dim_hidden_flat = dim_hidden

    def _get_kernel_and_mask(module):
        if hasattr(module, 'kernel'):
            return module.kernel, getattr(module, 'update_mask', None)
        if hasattr(module, 'W'):
            return module.W, getattr(module, '_mask', None)
        return None, None

    # Input
    win_mod = new_orch.lmap[1][0]
    kernel_win, mask_win = _get_kernel_and_mask(win_mod)
    if kernel_win is not None:
        rescale = cfg['optimizer']['weight_decay_win'] * cfg['optimizer']['learning_rate_win'] / (dim_hidden_flat ** 0.5)
        dW = kernel_win * rescale
        if mask_win is not None:
            dW = dW * mask_win
        new_orch = eqx.tree_at(lambda m: m.lmap[1][0].kernel, new_orch, kernel_win + dW)
    else:
        # fallback for FC
        W_in = new_orch.lmap[1][0].W
        rescale = cfg['optimizer']['weight_decay_win'] * cfg['optimizer']['learning_rate_win'] / (dim_hidden_flat ** 0.5)
        new_orch = eqx.tree_at(lambda m: m.lmap[1][0].W, new_orch, W_in + W_in * rescale)

    # J
    j_mod = new_orch.lmap[1][1]
    kernel_j, mask_j = _get_kernel_and_mask(j_mod)
    if kernel_j is not None:
        rescale = cfg['optimizer']['weight_decay_j'] * cfg['optimizer']['learning_rate_j'] / (dim_hidden_flat ** 0.5)
        dW = kernel_j * rescale
        if mask_j is not None:
            dW = dW * mask_j
        new_orch = eqx.tree_at(lambda m: m.lmap[1][1].kernel, new_orch, kernel_j + dW)
    else:
        J = new_orch.lmap[1][1].J if hasattr(new_orch.lmap[1][1], 'J') else None
        if J is not None:
            rescale = cfg['optimizer']['weight_decay_j'] * cfg['optimizer']['learning_rate_j'] / (dim_hidden_flat ** 0.5)
            new_orch = eqx.tree_at(lambda m: m.lmap[1][1].J, new_orch, J + J * rescale)

    # W_out (assume FC)
    W_out = new_orch.lmap[2][1].W
    rescale = cfg['optimizer']['weight_decay_wout'] * cfg['optimizer']['learning_rate_wout'] / (dim_hidden_flat ** 0.5)
    new_orch = eqx.tree_at(lambda m: m.lmap[2][1].W, new_orch, W_out + W_out * rescale)

    return new_orch

print('decay defined')


decay defined


In [24]:
# Quick driver: build a conv model and run a couple of train steps
cfg = copy.deepcopy(PARAMS)
cfg['model']['kwargs'].update({'win_type':'conv','j_type':'conv','kernel_size':3, 'c_in': 3})

state, orch = build_model(**cfg['model']['kwargs'])

ds = Cifar10(**cfg['data']['kwargs'])
key = jax.random.PRNGKey(cfg['master_seed'])
key, data_key = jax.random.split(key)
ds.build(data_key)

# Learning-rate map
lr_map = make_lr_map_v2(orch, overrides={(1,0):'w_in',(1,1):'j',(2,1):'w_out'})

opt = optax.multi_transform({
    'default': optax.sgd(0.0),
    'w_in': optax.sgd(cfg['optimizer']['learning_rate_win']),
    'j': optax.sgd(cfg['optimizer']['learning_rate_j']),
    'w_out': optax.sgd(cfg['optimizer']['learning_rate_wout']),
}, lr_map)
opt_state = opt.init(eqx.filter(orch, eqx.is_inexact_array))

trainer = DynamicalTrainer(orch, state, opt, opt_state, **cfg['trainer']['kwargs'])



# run a couple train steps
for epoch in range(2):
    for xb, yb in ds:
        key = trainer.train_step(xb, yb, key)
        trainer.orchestrator = decay(trainer.orchestrator, cfg)

# eval a batch
for xb, yb in ds.iter_test():
    key, metrics = trainer.eval_step(xb, yb, key)
    print('eval metrics:', metrics)
    break

print('Driver finished')

eval metrics: {'accuracy': Array(0.125, dtype=float32)}
Driver finished


In [8]:
history: dict[str, list[float]] = {
    "train_acc": [],
    "eval_acc": [],
}

for epoch in range(0, 5):
    # ---- Train ----
    if epoch != 0:
        for xb, yb in ds:
            key = trainer.train_step(xb, yb, key)
            trainer.orchestrator = decay(trainer.orchestrator, cfg)

    # ---- Eval on test split ----
    accs_eval = []
    for xb, yb in ds.iter_test():
        key, metrics = trainer.eval_step(xb, yb, key)
        accs_eval.append(metrics["accuracy"])

    acc_eval = float(jnp.mean(jnp.array(accs_eval))) if accs_eval else float("nan")

    # ---- Eval on train split ----
    accs_train = []
    for xb, yb in ds:
        key, metrics = trainer.eval_step(xb, yb, key)
        accs_train.append(metrics["accuracy"])
    acc_train = float(jnp.mean(jnp.array(accs_train))) if accs_train else float("nan")

    history["train_acc"].append(acc_train)
    history["eval_acc"].append(acc_eval)

    print(f"Epoch {epoch:03d} | train_acc={acc_train:.4f} | eval_acc={acc_eval:.4f}")


Epoch 000 | train_acc=0.8903 | eval_acc=0.8958
Epoch 001 | train_acc=0.8934 | eval_acc=0.8958
Epoch 002 | train_acc=0.9029 | eval_acc=0.9057
Epoch 003 | train_acc=0.9047 | eval_acc=0.9056
Epoch 004 | train_acc=0.9072 | eval_acc=0.9105
